# Bronze → Silver: leituras

**Fonte:** `venergy.bronze.leituras` (todas as colunas como texto) e `venergy.bronze.medidores` (cadastro de medidores).
**Destino:** `venergy.silver.leituras` (Delta).

**Pipeline (seção 5 de `docs/auditoria_dados/auditoria_leituras.md`), em ordem**
1. Remove duplicatas exatas sobre as 9 colunas de negócio (−21.523).
2. Descarta as leituras cujo `id_medidor` não existe em `venergy.bronze.medidores`, que são os clones `LTX`/`MD9…` (−8.609). Restam 2.152.336 leituras, uma por (`id_uc`, `mes_referencia`).
3. Normaliza `''`, `N/A`, `NULL` e `-1` em `leitura_anterior` para null, **antes** de converter para número.
4. Converte `data_leitura` pela expressão regular de cada valor: `yyyy-MM-dd`, `dd/MM/yyyy`, `dd.MM.yyyy` e hífen **obrigatoriamente** como `MM-dd-yyyy`.
5. Converte `consumo_kwh` (`-N.0` → inteiro) e inverte o sinal **só** das leituras de sinal invertido: `leitura_anterior` numérica, `leitura_anterior ≤ leitura_atual` e `consumo_kwh = −(atual − anterior)` exato.
6. Marca, **sem** inverter, **sem** estimar e **sem** `abs()`, mantendo o valor original:
   - `flag_troca_medidor`: consumo negativo e `leitura_anterior > leitura_atual` (mês da troca de medidor);
   - `flag_sem_leitura_anterior`: consumo negativo sem leitura anterior;
   - `flag_sinal_nao_confirmado`: consumo negativo com leituras crescentes, mas módulo diferente de `atual − anterior` em ±1 kWh.

   `flag_sinal_corrigido` identifica as leituras que tiveram o sinal invertido no passo 5.
7. Tipagem final: leituras e consumo como `INT`, `data_leitura` como `DATE`, `mes_referencia` mantido como `AAAA-MM`, mais `source_file` e `ingestion_datetime` herdados da Bronze.

**Decisões desta camada**
- A diferença de ±1 kWh entre `consumo_kwh` e `atual − anterior` é mantida (arredondamento). `consumo_kwh` é a fonte de verdade do consumo.
- A estimativa do consumo nos meses com flag (pela fatura ou pela média da UC) fica para decisão do time. Para somar consumo, filtre `consumo_kwh > 0` ou exclua as linhas com flag.
- Carga completa com `overwrite`, e todas as validações rodam **antes** de gravar: uma Silver inválida nunca substitui a atual.

Regras, contagens e pendências em `docs/silver/silver_leituras.md`.

In [ ]:
catalog = "venergy"
bronze_schema = "bronze"
silver_schema = "silver"

bronze_leituras = f"{catalog}.{bronze_schema}.leituras"
bronze_medidores = f"{catalog}.{bronze_schema}.medidores"
silver_table = f"{catalog}.{silver_schema}.leituras"

COLUNAS_NEGOCIO = [
    "id_leitura", "id_uc", "id_medidor", "data_leitura", "mes_referencia",
    "leitura_anterior", "leitura_atual", "consumo_kwh", "tipo_leitura",
]
FLAGS = ["flag_sinal_corrigido", "flag_troca_medidor", "flag_sem_leitura_anterior", "flag_sinal_nao_confirmado"]
SILVER_COLUMNS = COLUNAS_NEGOCIO + FLAGS + ["source_file", "ingestion_datetime"]

AUSENCIAS_LEITURA_ANTERIOR = ["", "N/A", "NULL", "-1"]

# Formato de data por expressão regular. O hífen é sempre MM-dd-yyyy (auditoria, seção 3.1).
FORMATOS_DATA = [
    (r"^\d{4}-\d{2}-\d{2}$", "yyyy-MM-dd"),
    (r"^\d{2}/\d{2}/\d{4}$", "dd/MM/yyyy"),
    (r"^\d{2}\.\d{2}\.\d{4}$", "dd.MM.yyyy"),
    (r"^\d{2}-\d{2}-\d{4}$", "MM-dd-yyyy"),
]

EXPECTED_BRONZE = 2_182_468
EXPECTED_SEM_DUPLICATAS = 2_160_945  # −21.523 duplicatas exatas
EXPECTED_SILVER = 2_152_336          # −8.609 clones LTX com medidor inexistente

# Contagens das flags sobre a base limpa (após os passos 1 e 2). No bruto eram 1.349, 6.327, 115 e 359:
# as diferenças são as duplicatas exatas e os clones LTX dessas linhas (ver docs/silver/silver_leituras.md).
EXPECTED_FLAGS = {
    "flag_sinal_corrigido": 1_334,
    "flag_troca_medidor": 6_215,
    "flag_sem_leitura_anterior": 114,
    "flag_sinal_nao_confirmado": 353,
}

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

## 1–2. Duplicatas exatas e leituras com medidor inexistente

In [ ]:
from pyspark.sql import functions as F

df_bronze = spark.table(bronze_leituras)

# 1. Duplicatas exatas: as cópias estão sempre no mesmo arquivo, então source_file não muda
df_sem_duplicatas = df_bronze.dropDuplicates(COLUNAS_NEGOCIO)

# 2. Clones LTX: medidor (MD9…) inexistente no cadastro
medidores_cadastrados = spark.table(bronze_medidores).select("id_medidor").distinct()
df_limpo = df_sem_duplicatas.join(medidores_cadastrados, on="id_medidor", how="left_semi")

contagem = {
    "bronze": df_bronze.count(),
    "sem_duplicatas": df_sem_duplicatas.count(),
    "sem_medidor_inexistente": df_limpo.count(),
}
print(contagem)
assert contagem["bronze"] == EXPECTED_BRONZE, f"Bronze com {contagem['bronze']} linhas"
assert contagem["sem_duplicatas"] == EXPECTED_SEM_DUPLICATAS, f"Após remover duplicatas: {contagem['sem_duplicatas']}"
assert contagem["sem_medidor_inexistente"] == EXPECTED_SILVER, f"Após remover clones: {contagem['sem_medidor_inexistente']}"

## 3–7. Ausências, datas, tipagem, sinal e flags

In [ ]:
# 3. Ausências de leitura_anterior viram null ANTES do cast (o -1 passaria como número válido)
anterior_txt = (
    F.when(F.col("leitura_anterior").isNull() | F.col("leitura_anterior").isin(AUSENCIAS_LEITURA_ANTERIOR), None)
    .otherwise(F.col("leitura_anterior"))
)

# 4. Data pelo formato identificado na regex de cada valor
data = None
for regex, formato in FORMATOS_DATA:
    convertida = F.to_date(F.expr(f"try_to_timestamp(data_leitura, '{formato}')"))
    data = (F.when if data is None else data.when)(F.col("data_leitura").rlike(regex), convertida)

df_tipado = (
    df_limpo
    .withColumn("anterior_txt", anterior_txt)
    .withColumn("ant", F.expr("try_cast(anterior_txt AS INT)"))
    .withColumn("atu", F.expr("try_cast(leitura_atual AS INT)"))
    # 5. -N.0 → inteiro (a parte decimal é sempre .0, validado abaixo)
    .withColumn("cons_txt", F.regexp_replace("consumo_kwh", r"\.0$", ""))
    .withColumn("cons", F.expr("try_cast(cons_txt AS INT)"))
    .withColumn("dt", data)
)

negativo = F.col("cons") < 0
crescente = F.col("ant").isNotNull() & (F.col("ant") <= F.col("atu"))
modulo_exato = F.col("cons") == -(F.col("atu") - F.col("ant"))

df_flags = (
    df_tipado
    .withColumn("flag_sinal_corrigido", F.coalesce(negativo & crescente & modulo_exato, F.lit(False)))
    .withColumn("flag_troca_medidor", F.coalesce(negativo & F.col("ant").isNotNull() & (F.col("ant") > F.col("atu")), F.lit(False)))
    .withColumn("flag_sem_leitura_anterior", F.coalesce(negativo & F.col("ant").isNull(), F.lit(False)))
    .withColumn("flag_sinal_nao_confirmado", F.coalesce(negativo & crescente & ~modulo_exato, F.lit(False)))
)

# 6–7. Só o sinal invertido confirmado é corrigido; as demais negativas mantêm o valor original
df_silver = df_flags.select(
    "id_leitura",
    "id_uc",
    "id_medidor",
    F.col("dt").alias("data_leitura"),
    "mes_referencia",
    F.col("ant").alias("leitura_anterior"),
    F.col("atu").alias("leitura_atual"),
    F.when(F.col("flag_sinal_corrigido"), -F.col("cons")).otherwise(F.col("cons")).alias("consumo_kwh"),
    "tipo_leitura",
    *FLAGS,
    "source_file",
    "ingestion_datetime",
)

## Validação antes de gravar
Qualquer divergência interrompe o notebook aqui e a Silver atual não é tocada.

In [ ]:
flag_negativa = F.col("flag_troca_medidor") | F.col("flag_sem_leitura_anterior") | F.col("flag_sinal_nao_confirmado")
qtd_flags = sum(F.col(f).cast("int") for f in FLAGS)
conta = lambda condicao: F.sum(F.when(condicao, 1).otherwise(0))

# Conferência dos textos de origem (antes da tipagem)
checagem_origem = df_flags.agg(
    conta(F.col("anterior_txt").isNotNull() & F.col("ant").isNull()).alias("anterior_nao_numerico"),
    conta(F.col("atu").isNull()).alias("atual_nula"),
    conta(~F.col("consumo_kwh").rlike(r"^-?\d+(\.0)?$") | F.col("cons").isNull()).alias("consumo_invalido"),
    conta(F.col("flag_sinal_nao_confirmado") & (F.abs(-F.col("cons") - (F.col("atu") - F.col("ant"))) != 1)).alias("nao_confirmado_fora_de_1kwh"),
).first().asDict()

checagem_silver = df_silver.agg(
    F.count("*").alias("linhas"),
    F.countDistinct("id_uc", "mes_referencia").alias("pares_uc_mes"),
    conta(F.col("data_leitura").isNull()).alias("data_nula"),
    conta(F.date_format("data_leitura", "yyyy-MM") != F.col("mes_referencia")).alias("mes_divergente"),
    conta(~F.col("mes_referencia").rlike(r"^\d{4}-(0[1-9]|1[0-2])$")).alias("mes_referencia_invalido"),
    conta((F.col("consumo_kwh") < 0) & ~flag_negativa).alias("negativo_sem_flag"),
    conta(qtd_flags > 1).alias("mais_de_uma_flag"),
    *[conta(F.col(f)).alias(f) for f in FLAGS],
).first().asDict()

print(checagem_origem)
print(checagem_silver)

assert all(v == 0 for v in checagem_origem.values()), f"Valores de origem fora do esperado: {checagem_origem}"
assert checagem_silver["linhas"] == EXPECTED_SILVER, f"Silver com {checagem_silver['linhas']} linhas"
assert checagem_silver["pares_uc_mes"] == EXPECTED_SILVER, "Mais de uma leitura por (id_uc, mes_referencia)"
assert checagem_silver["data_nula"] == 0, "Há data_leitura nula"
assert checagem_silver["mes_divergente"] == 0, "Mês da data diverge de mes_referencia"
assert checagem_silver["mes_referencia_invalido"] == 0, "mes_referencia fora de AAAA-MM"
assert checagem_silver["negativo_sem_flag"] == 0, "Consumo negativo sem flag"
assert checagem_silver["mais_de_uma_flag"] == 0, "Linha com mais de uma flag"
flags = {f: checagem_silver[f] for f in FLAGS}
assert flags == EXPECTED_FLAGS, f"Contagem das flags diverge: {flags}"

In [ ]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

In [ ]:
# Depois de gravar: colunas do contrato e volume esperado
tabela = spark.table(silver_table)
linhas_tabela = tabela.count()

assert tabela.columns == SILVER_COLUMNS, f"Colunas inesperadas: {tabela.columns}"
assert linhas_tabela == EXPECTED_SILVER, f"Tabela com {linhas_tabela} linhas, esperado {EXPECTED_SILVER}"
print("OK: 2.152.336 linhas gravadas em", silver_table)